Import libraries

In [1]:
import pandas as pd
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from ragas import evaluate
from ragas.metrics import (
    answer_relevancy,
    context_precision,
    context_recall,
    faithfulness,
)

/Users/blessingmagabane/Library/Python/3.9/lib/python/site-packages/google/ai/generativelanguage_v1beta/__init__.py:295: FutureWarning: You are using a Python version (3.9.6) which Google will stop supporting in google.ai.generativelanguage_v1beta in January 2026. Please upgrade to the latest Python version, or at least to Python 3.10, before then, and then update google.ai.generativelanguage_v1beta.
  warnings.warn(
/Users/blessingmagabane/Library/Python/3.9/lib/python/site-packages/torch/utils/_pytree.py:185: FutureWarning: optree is installed but the version is too old to support PyTorch Dynamo in C++ pytree. C++ pytree support is disabled. Please consider upgrading optree using `python3 -m pip install --upgrade 'optree>=0.13.0'`.
  warnings.warn(
/var/folders/dr/mrkz6zcx1tz_z87q4dtf9qym0000gn/T/ipykernel_94840/87409815.py:6: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. 

Question 1

In [ ]:
# Initialize judge LLM and embedding model
eval_llm = llm
eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "What are all direct incoming and outgoing relationships centered on the Permian Region node?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [[
            "Americas  EuropeAsia OceanianTotal OECD   NON-OECD SUPPLY Europe  China  Middle East "
    ]],
    "reference": [
    "Worldwide Delivery Bottlenecks & Brent (International Petroleum Standards): Brent crude (together with alternative petroleum standards such as North Sea Dated, WTI, and Dubai) is categorized alongside international distribution interruptions and bottlenecks. In particular, the passage emphasizes that the effective blockade of the Strait of Hormuz (a primary global petroleum transport bottleneck) and armed conflicts fuel increased instability and immediately affect the Brent crude petroleum spot valuation.National Extraction Regions & Henry Hub (American Natural Gas Standard): Henry Hub is linked to internal extraction areas and localized facilities. The document observes that national extraction zones close to the Gulf Coast LNG shipping terminals generate sufficient natural gas to maintain stockpiles higher than the half-decade mean, which constrains rising forces on Henry Hub natural gas costs."
    ],
}

graph_rag_data = {
    "question": [
        "What are all direct incoming and outgoing relationships centered on the Permian Region node?"
    ],
    "answer": [
       "Based on the provided context, which consists of tabular data, there are no explicitly defined graph-based incoming or outgoing structural relationships. However, we can identify the relationships based on the groupings and numerical data (positive and negative flows) associated with the Permian region node. For peer group relationships (sibling nodes), the Permian region belongs to the same category/grouping as the following regional nodes: Bakken region, Eagle Ford region, and Haynesville region."
    ],
    "contexts": 
        [[
            "The direct incoming and outgoing relationships centered on the Permian Region node are outgoing relationships (from Permian Region), including LOCATED_IN United States and Lower 48 States, AFFECTS_SUPPLY_OF Total Primary Supply, and PRODUCES Natural Gas. Incoming relationships (to Permian Region) include Cumulative Drilled But Uncompleted Wells is LOCATED_IN the Permian Region, New Wells Drilled and New Wells Drilled Per Rig are MEASURED_IN the Permian Region, U.S. Energy Information Administration FORECASTS_OUTLOOK for the Permian Region, and Crude Oil and Crude Oil Production From Newly Completed Wells are PRODUCED_IN the Permian Region."
    ]],

    "reference": [
      "Worldwide Delivery Bottlenecks & Brent (International Petroleum Standards): Brent crude (together with alternative petroleum standards such as North Sea Dated, WTI, and Dubai) is categorized alongside international distribution interruptions and bottlenecks. In particular, the passage emphasizes that the effective blockade of the Strait of Hormuz (a primary global petroleum transport bottleneck) and armed conflicts fuel increased instability and immediately affect the Brent crude petroleum spot valuation.National Extraction Regions & Henry Hub (American Natural Gas Standard): Henry Hub is linked to internal extraction areas and localized facilities. The document observes that national extraction zones close to the Gulf Coast LNG shipping terminals generate sufficient natural gas to maintain stockpiles higher than the half-decade mean, which constrains rising forces on Henry Hub natural gas costs."
    ],

}

In [6]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
        raise_exceptions=True
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df


In [7]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG      0.000000          0.932215                0.0             0.0
Vector RAG      0.285714          0.000000                0.0             0.0


Questions 2

In [8]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How are the pricing benchmarks (Brent and Henry Hub) grouped into graph communities relative to global supply chokepoints versus domestic production basins?"
    ],
    "answer": [
       "Global Supply Chokepoints & Brent (Global Oil Benchmarks): Brent crude (along with other crude benchmarks like North Sea Dated, WTI, and Dubai) is grouped with global supply disruptions and chokepoints. Specifically, the text highlights that the de facto closure of the Strait of Hormuz (a major world oil transit chokepoint) and military action drive heightened volatility and directly impact the Brent crude oil spot price. Domestic Production Basins & Henry Hub (U.S. Natural Gas Benchmark): Henry Hub is grouped with domestic production basins and regional infrastructure. The text notes that domestic production regions near the Gulf Coast LNG export facilities produce enough natural gas to keep inventories above the five-year average, which limits upward pressure on Henry Hub natural gas prices."
    ],
    "contexts": [[
        "U.S. net trade of hydrocarbon gas liquids (HGL) million barrels per day net trade propane ethane natural gasoline butanes forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026 net imports net exports Henry Hub natural gas price and NYMEX futures price dollars per million British thermal units Note: Futures curve is the average settlement price for five trading days ending May 7, 2026.  Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026, Bloomberg L.P., and LSEG Data STEO forecast NYMEX futures price Henry Hub spot price U.S. natural gas prices dollars per thousand cubic feet forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026"
    ]],  
    "reference": [
        "In knowledge graphs modeled on energy supply chains, graph community detection algorithms (such as Louvain or Leiden modularity maximization) partition pricing benchmarks into distinct structural clusters based on directional edge density, physical commodity movement, and price correlation strength. 1. The Global Seaborne Trade Community (Brent Crude),community Alignment: Brent is assigned to the Global Maritime & Geopolitical Trade Cluster. Integration with Chokepoints: Nodes representing major global supply chokepoints (Strait of Hormuz, Bab el-Mandeb, Suez Canal, Strait of Malacca) exhibit tight structural coupling and high edge-weights with the Brent crude node. Network Dynamics, edge drivers, Physical VLCC/Suezmax tanker traffic paths, geopolitical risk premiums, arbitrage differentials (e.g., Brent-Dubai, Brent-WTI), and sovereign export quotas (OPEC+). Node Role: Brent acts as a high-betweenness central hub within the global cluster, propagating macro-geopolitical shocks and maritime transit disruptions across international crude export nodes. 2. The Domestic North American Pipeline Community (Henry Hub). Community Alignment: Henry Hub is assigned to the North American Landlocked Energy & Infrastructure Cluster. Integration with Production Basins: Henry Hub shares high intra-community edge density with domestic onshore supply basins (Permian Basin, Marcellus Shale, Haynesville Shale) and regional pipeline/storage hubs (Waha Hub, Katy Hub, Columbia Gas). Network Dynamics, Edge Drivers: Contractual pipeline capacity constraints, physical natural gas gathering/processing flows, regional basis spreads, domestic utility power burn, and underground storage injection/withdrawal cycles. Node Role: Henry Hub functions as the primary market-clearing node for North American pipeline gas, isolated from physical seaborne crude dynamics by regional physical transport infrastructure constraints. Community Boundary Interconnections While community partition algorithms segregate Brent and Henry Hub into distinct macro-clusters, specific bridge entities establish structural cross-community edges: LNG Liquefaction Export Facilities: Terminals such as Sabine Pass, Freeport, and Corpus Christi LNG form direct cross-community bridge edges, converting domestic pipeline gas tied to Henry Hub into seaborne cargoes linked to global energy pricing (e.g., TTF, JKM, and Brent-indexed contracts). Dual-Commodity Basins: Production nodes like the Permian Basin maintain dual edges across both communities—exporting crude oil that integrates into the Brent-linked seaborne trade cluster while delivering associated natural gas that flows directly into the Henry Hub pipeline network."
    ],
}

graph_rag_data = {
    "question": [
        "How are the pricing benchmarks (Brent and Henry Hub) grouped into graph communities relative to global supply chokepoints versus domestic production basins?"
    ],
    "answer": [
       "Based on the provided data, the pricing benchmarks Brent and Henry Hub are connected to the domestic production basin of the Gulf Of America, while there are no connections associated with global supply chokepoints. Specifically, their relationships in the graph are structured as follows: Brent is linked to the Gulf Of America basin through pathways involving OPEC (which forecasts and affects the supply of crude oil) and the North Sea (where Brent is located and produced), while Henry Hub is connected to the Gulf Of America basin through domestic pathways, specifically via its location in the United States (U.S.), which is linked to the production of crude oil and lease condensate in the Gulf Of America."
    ],
    "contexts": 
        [["The Brent benchmark is associated with the Gulf Of America basin region and has no listed chokepoints. The Henry Hub benchmark is associated with the Gulf Of America basin region and has no listed chokepoints."]],
    
    "reference": [
        "Worldwide Delivery Bottlenecks & Brent (International Petroleum Standards): Brent crude (together with alternative petroleum standards such as North Sea Dated, WTI, and Dubai) is categorized alongside international distribution interruptions and bottlenecks. In particular, the passage emphasizes that the effective blockade of the Strait of Hormuz (a primary global petroleum transport bottleneck) and armed conflicts fuel increased instability and immediately affect the Brent crude petroleum spot valuation.National Extraction Regions & Henry Hub (American Natural Gas Standard): Henry Hub is linked to internal extraction areas and localized facilities. The document observes that national extraction zones close to the Gulf Coast LNG shipping terminals generate sufficient natural gas to maintain stockpiles higher than the half-decade mean, which constrains rising forces on Henry Hub natural gas costs."

    ],
}

In [9]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [10]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.0          0.934190                0.0             0.0
Vector RAG           0.0          0.868126                0.0             0.0


Question 3

In [11]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "Which nodes exhibit the highest degree centrality regarding global crude oil price propagation?"
    ],
    "answer": [
       "Based on the provided context, there is no mention of degree centrality or nodes involved in global crude oil price propagation. Therefore, I do not know the answer to this question from the given text."

    ],
    "contexts": [[
            "Affected by the coronavirus disease 2019 (COVID-19) pandemic and the oil price war between Russia and Saudi Arabia from February to March 2020, the oil price represents a total reduction of 55.57%. The collapse in crude oil price and the decline in oil demand caused by the outbreak of the COVID-19 pandemic have sharply reduced petroleum production capacity in the U.S. Oil consumption has dropped signi ﬁcantly since the beginning of the lockdown measures adopted by the U.S. government. With the reopening policy released in May 2020, oil consumption saw a rapid recovery. Consequently, forecasting oil price, production, and consumption becomes challenging. Fortunately, as social media messages contain explanations or analyses of the relevant restrictions or reopening policies, the utilization of online oil news can accurately predict the large ﬂuctuations in oil price, production, and consumption during the COVID-19 pandemic."

    ]],
    "reference": [

        "In global crude oil knowledge graphs, the nodes exhibiting the highest degree centrality regarding price propagation are international benchmark pricing hubs, critical maritime trade chokepoints, primary export infrastructure nodes, and centralized supply-policy organizations. Brent Crude Benchmark (ICE Brent): Holds the highest overall degree centrality (both in-degree from physical crude assessments and out-degree to derivative contracts), directly referencing and setting price differentials for over 60% to 70% of global physical seaborne crude trade streams. Strait of Hormuz: Possesses the highest degree centrality among physical transit and infrastructure nodes, intersecting supply edges from major Middle Eastern oil producers (Saudi Arabia, UAE, Iraq, Kuwait, Iran) and outbound shipping edges to global import terminals in Europe, Asia, and North America. OPEC+ Ministerial Secretariat / Saudi Aramco: Exhibits dominant direct out-degree centrality, maintaining direct policy and operational edges connected to sovereign production quota nodes, baseline capacities, and monthly Official Selling Price (OSP) adjustments. U.S. Gulf Coast Export Terminals (PADD 3 / Corpus Christi, Houston, LOOP): Functions as a primary structural bridge node with high in-degree connectivity from onshore pipeline networks (Permian, Eagle Ford) and high out-degree connectivity to international seaborne markets. West Texas Intermediate (WTI) / Cushing Hub: Maintains high degree centrality connecting North American physical gatherers, storage operators, financial hedging contracts (NYMEX), and transpacific/transatlantic export pricing edges."
    ],
}

graph_rag_data = {
    "question": [
        "Which nodes exhibit the highest degree centrality regarding global crude oil price propagation?"
    ],
    "answer": [
       "The nodes that exhibit the highest degree centrality regarding global crude oil price propagation are North Sea Dated (with a degree centrality of 18), Opec Reference Basket (12), and Crude Oil (9). Other notable nodes in the network include Refining Margins (8), Gasoline (7), Dubai (7), Wti (7), Nymex Wti (7), Gold (7), and Ice Brent (6)."

    ],
    "contexts": 
        [[
            "In the knowledge graph, North Sea Dated has a connection degree of 18. Opec Reference Basket has a connection degree of 12. Crude Oil has a connection degree of 9. Refining Margins has a connection degree of 8. Gasoline, Dubai, Wti, Nymex Wti, and Gold each have a connection degree of 7. Ice Brent has a connection degree of 6."

    ]],
    "reference": [

        "In global crude oil knowledge graphs, the nodes exhibiting the highest degree centrality regarding price propagation are international benchmark pricing hubs, critical maritime trade chokepoints, primary export infrastructure nodes, and centralized supply-policy organizations. Brent Crude Benchmark (ICE Brent): Holds the highest overall degree centrality (both in-degree from physical crude assessments and out-degree to derivative contracts), directly referencing and setting price differentials for over 60% to 70% of global physical seaborne crude trade streams. Strait of Hormuz: Possesses the highest degree centrality among physical transit and infrastructure nodes, intersecting supply edges from major Middle Eastern oil producers (Saudi Arabia, UAE, Iraq, Kuwait, Iran) and outbound shipping edges to global import terminals in Europe, Asia, and North America. OPEC+ Ministerial Secretariat / Saudi Aramco: Exhibits dominant direct out-degree centrality, maintaining direct policy and operational edges connected to sovereign production quota nodes, baseline capacities, and monthly Official Selling Price (OSP) adjustments. U.S. Gulf Coast Export Terminals (PADD 3 / Corpus Christi, Houston, LOOP): Functions as a primary structural bridge node with high in-degree connectivity from onshore pipeline networks (Permian, Eagle Ford) and high out-degree connectivity to international seaborne markets. West Texas Intermediate (WTI) / Cushing Hub: Maintains high degree centrality connecting North American physical gatherers, storage operators, financial hedging contracts (NYMEX), and transpacific/transatlantic export pricing edges."
    ],
}

In [12]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [13]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.5               1.0                0.0             0.0
Vector RAG           1.0               0.0                0.0             0.0


Question 4. --- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How does GraphRAG reconcile opposing pricing pressures when summer cooling demand increases while associated natural gas production simultaneously rises?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "How does GraphRAG reconcile opposing pricing pressures when summer cooling demand increases while associated natural gas production simultaneously rises?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'demand': {'id': 'U.S. Summer Cooling Degree Days'}, 'r1': ({'id': 'U.S. Summer Cooling Degree Days'}, 'AFFECTS', {'id': 'Natural Gas'}), 'price': {'id': 'Natural Gas'}, 'r2': ({'id': 'Natural Gas'}, 'AFFECTS', {'id': 'Natural Gas-Weighted Manufacturing'}), 'gas': {'id': 'Natural Gas-Weighted Manufacturing'}}, {'demand': {'id': 'International Demand'}, 'r1': ({'id': 'International Demand'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'price': {'id': 'Crude Oil'}, 'r2': ({'id': 'Natural Gas Liquids'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'gas': {'id': 'Natural Gas Liquids'}}, {'demand': {'id': 'World Demand'}, 'r1': ({'id': 'World Demand'}, 'AFFECTS', {'id': 'Crude Oil'}), 'price': {'id': 'Crude Oil'}, 'r2': ({'id': 'Natural Gas Liquids'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'gas': {'id': 'Natural Gas Liquids'}}, {'demand': {'id': 'International Demand'}, 'r1': ({'id': 'International Demand'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'price': {'id': 'Crude Oil'}, 'r2': ({'id': 'Natural Gasoline'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'gas': {'id': 'Natural Gasoline'}}, {'demand': {'id': 'World Demand'}, 'r1': ({'id': 'World Demand'}, 'AFFECTS', {'id': 'Crude Oil'}), 'price': {'id': 'Crude Oil'}, 'r2': ({'id': 'Natural Gasoline'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}), 'gas': {'id': 'Natural Gasoline'}}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question -- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "Which nodes exhibit the highest degree centrality regarding global crude oil price propagation?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "Which nodes exhibit the highest degree centrality regarding global crude oil price propagation?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'n.id': 'North Sea Dated', 'degree': 18}, {'n.id': 'Opec Reference Basket', 'degree': 12}, {'n.id': 'Crude Oil', 'degree': 9}, {'n.id': 'Refining Margins', 'degree': 8}, {'n.id': 'Gasoline', 'degree': 7}, {'n.id': 'Dubai', 'degree': 7}, {'n.id': 'Wti', 'degree': 7}, {'n.id': 'Nymex Wti', 'degree': 7}, {'n.id': 'Gold', 'degree': 7}, {'n.id': 'Ice Brent', 'degree': 6}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question 6

In [14]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How are cluster communities formed around international crude oil markets versus domestic U.S. energy assets?"
    ],
    "answer": [
       "Based on the provided context, there is no mention of cluster communities or how they are formed around international crude oil markets versus domestic U.S. energy assets. However, the text does distinguish between international and domestic influences using text mining of news keywords. For international crude oil markets, online news collections with international keywords like Crude oil and Crude oil price are conducive to forecasting oil prices, production, and inventory, indicating these factors are influenced by various international events. For domestic U.S. energy assets, news collections with domestic keywords like American oil facilitate consumption forecasting, indicating that oil consumption is primarily influenced by domestic events."
    ],
    "contexts": [[
            "reserves outside the producing country improves resilience for both. For consuming countries, hosting crude owned by major exporters creates an immediately accessible buffer against supply  disruptions, shortens response times in a crisis, and supplements strategic stocks without requiring  the state to finance the additional inventory itself. For producers, overseas storage places barrels  closer to end-users, improves optionality in destination sales, supports in-tank transfers and prompt  deliveries, and strengthens market share in key importing regions. Typically, the producer can use  the storage commercially in normal times, while the host country receives priority access during emergencies.  Japan dominates this model, hosting joint stockpiling with Saudi Aramco, which includes 8.2 mb in 13 tanks at the Okinawa CTS base as well as a commercial arrangement with ENEOS for about 3 mb"

    ]],
    "reference": [
     "Cluster communities within energy knowledge graphs are partitioned by community detection algorithms (such as Louvain or Leiden modularity optimization) that group nodes sharing high internal edge density, strong transaction velocity, and tightly coupled pricing correlations. International Crude Market Clusters International communities form around global supply chain dynamics where physical transport is unconstrained by landlocked pipelines. Maritime Transit Coupling: Edges connect exporting nations, charter shipping lines, and major import hubs via maritime chokepoints (Strait of Hormuz, Bab el-Mandeb, Malacca Strait). A risk event at a transit node rapidly propagates across all member nodes in the global cluster. Geopolitical & Fiscal Alignment: Sovereign producers (e.g., Saudi Arabia, UAE, Iraq) form tight sub-communities linked by coordinated export target edges and shared state revenue dependence on global crude benchmarks. Global Derivative Interconnections: Financial hedge contracts, spot price differentials (e.g., Brent-WTI spread), and seaborne freight rates establish dense multi-hop edges linking international refining hubs in Asia and Europe directly to the Brent benchmark node. Domestic U.S. Energy Asset Clusters Domestic U.S. communities form around rigid physical infrastructure topologies, where localized supply-demand balances create highly distinct sub-graphs. Basin-Specific Sub-Communities: Onshore assets cluster tightly by geographical basin boundaries (e.g., Permian Basin, Marcellus Shale, Haynesville Shale). Edge relationships are driven by field-level gathering lines, rig counts, and local processing infrastructure. Pipeline Capacity Bottlenecks: Nodes within a basin remain clustered together until takeaway pipelines cross regional boundaries. When takeaway pipelines reach maximum capacity, the localized cluster becomes financially decoupled from national benchmarks, causing extreme basis spread divergences (e.g., Waha Hub negative pricing events). Power Sector & End-Use Integration: Domestic gas and power clusters feature direct edges between regional pipeline hubs, local distribution companies (LDCs), and regional electric transmission operators (e.g., ERCOT, PJM), tying commodity flows directly to localized weather and generation dispatch."
    ],
}

graph_rag_data = {
    "question": [
        "How are cluster communities formed around international crude oil markets versus domestic U.S. energy assets?"
    ],
    "answer": [
       "Based on the provided data, cluster communities are formed by linking regional crude sources, such as Alaskan Crude Oil (located in Alaska and Padd 5), to domestic U.S. energy assets like Refineries, Biofuels Plants, Us Refineries, and Coke Plants located within the United States. These domestic assets and regional clusters then connect to international crude oil markets through the United States supplying to Global Refineries."
    ],
    "contexts": 
        [[
            "North Sea Dated is priced against Dubai Spot. Refinery is located in U.S. Refineries are located in District Of Columbia. Refineries are located in 50 States. Petrochemical Plants are located in District Of Columbia. Petrochemical Plants are located in 50 States. Mtbe Plants are located in District Of Columbia. Mtbe Plants are located in 50 States. Russian Refineries are located in Russia. China's Petrochemical Plants are located in China. Habshan-Fujairah Pipeline is located in Uae."
    ]],
    "reference": [
     "Cluster communities within energy knowledge graphs are partitioned by community detection algorithms (such as Louvain or Leiden modularity optimization) that group nodes sharing high internal edge density, strong transaction velocity, and tightly coupled pricing correlations. International Crude Market Clusters International communities form around global supply chain dynamics where physical transport is unconstrained by landlocked pipelines. Maritime Transit Coupling: Edges connect exporting nations, charter shipping lines, and major import hubs via maritime chokepoints (Strait of Hormuz, Bab el-Mandeb, Malacca Strait). A risk event at a transit node rapidly propagates across all member nodes in the global cluster. Geopolitical & Fiscal Alignment: Sovereign producers (e.g., Saudi Arabia, UAE, Iraq) form tight sub-communities linked by coordinated export target edges and shared state revenue dependence on global crude benchmarks. Global Derivative Interconnections: Financial hedge contracts, spot price differentials (e.g., Brent-WTI spread), and seaborne freight rates establish dense multi-hop edges linking international refining hubs in Asia and Europe directly to the Brent benchmark node. Domestic U.S. Energy Asset Clusters Domestic U.S. communities form around rigid physical infrastructure topologies, where localized supply-demand balances create highly distinct sub-graphs. Basin-Specific Sub-Communities: Onshore assets cluster tightly by geographical basin boundaries (e.g., Permian Basin, Marcellus Shale, Haynesville Shale). Edge relationships are driven by field-level gathering lines, rig counts, and local processing infrastructure. Pipeline Capacity Bottlenecks: Nodes within a basin remain clustered together until takeaway pipelines cross regional boundaries. When takeaway pipelines reach maximum capacity, the localized cluster becomes financially decoupled from national benchmarks, causing extreme basis spread divergences (e.g., Waha Hub negative pricing events). Power Sector & End-Use Integration: Domestic gas and power clusters feature direct edges between regional pipeline hubs, local distribution companies (LDCs), and regional electric transmission operators (e.g., ERCOT, PJM), tying commodity flows directly to localized weather and generation dispatch."
    ]    
}

In [15]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [16]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG          0.00          0.953733                0.0             0.0
Vector RAG          0.25          0.000000                0.0             0.0


Question 7

In [17]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "What entity serves as the primary bridge node between the petroleum supply chain and the electric power sector?"
    ],
    "answer": [
       "Based on the provided context, there is no mention of a primary bridge node or any specific entity serving as a bridge between the petroleum supply chain and the electric power sector. Therefore, the provided text does not contain the answer to this question."
    ],
    "contexts": [[
            "Crude oil product supplied Secondary supply chain  Tertiary stocks of natural gas liquids Components of Supply and Disposition  Balance of supply and disposition  Components of supply"
    ]],
    "reference": [
        "Natural Gas Processing Facilities, Dual-Fuel Power Plants, and Petroleum Refineries serve as the primary structural bridge nodes linking the petroleum supply chain to the electric power sector.Among these, Natural Gas Processing & Gathering Hubs act as the principal volume bridge because they collect raw associated gas generated from upstream oil wells, strip out impurities and liquid hydrocarbons, and supply dry pipeline gas directly to natural gas-fired power generation plants.Key Bridging Node FunctionsNatural Gas Processing & Gathering Infrastructure:Petroleum Connection (Incoming): Receives raw associated gas produced alongside crude oil from upstream petroleum wells.Electric Power Connection (Outgoing): Delivers processed dry methane ($CH_4$) through pipeline networks directly to natural gas combined-cycle (NGCC) power plants and peaker units.Dual-Fuel Power Plants:Petroleum Connection (Incoming): Maintains supply lines and fuel storage for distillate fuel oil, diesel, or residual fuel oil (HFO).Electric Power Connection (Outgoing): Dispatches electricity to regional power grids during extreme weather events or natural gas supply disruptions by switching fuel sources.Petroleum Refineries (Residual Fuels & Petcoke):Petroleum Connection (Incoming): Refines crude oil streams into lighter transportation fuels while outputting heavy residual bottom fractions.Electric Power Connection (Outgoing): Supplies petroleum coke (petcoke) and heavy fuel oil directly to thermal power plants and industrial co-generation units.Electrified Oilfield & Midstream Infrastructure:Electric Power Connection (Incoming): Consumes bulk electricity drawn directly from regional transmission grids (e.g., ERCOT, SPP).Petroleum Connection (Outgoing): Powers electric drilling rigs, hydraulic fracturing fleets, and pipeline pump/compressor stations required to keep crude oil moving through midstream networks."

    ],
}

graph_rag_data = {
    "question": [
       "What entity serves as the primary bridge node between the petroleum supply chain and the electric power sector?"
    ],
    "answer": [
       "The United States and the U.S. Energy Information Administration serve as the primary bridge nodes between the petroleum supply chain and the electric power sector."

    ],
    "contexts": 
        [[
        "In the knowledge graph, the entity Refineries acts as a key bridge node with a strength score of 63."
    ]],
    "reference": [
        "Natural Gas Processing Facilities, Dual-Fuel Power Plants, and Petroleum Refineries serve as the primary structural bridge nodes linking the petroleum supply chain to the electric power sector.Among these, Natural Gas Processing & Gathering Hubs act as the principal volume bridge because they collect raw associated gas generated from upstream oil wells, strip out impurities and liquid hydrocarbons, and supply dry pipeline gas directly to natural gas-fired power generation plants.Key Bridging Node FunctionsNatural Gas Processing & Gathering Infrastructure:Petroleum Connection (Incoming): Receives raw associated gas produced alongside crude oil from upstream petroleum wells.Electric Power Connection (Outgoing): Delivers processed dry methane ($CH_4$) through pipeline networks directly to natural gas combined-cycle (NGCC) power plants and peaker units.Dual-Fuel Power Plants:Petroleum Connection (Incoming): Maintains supply lines and fuel storage for distillate fuel oil, diesel, or residual fuel oil (HFO).Electric Power Connection (Outgoing): Dispatches electricity to regional power grids during extreme weather events or natural gas supply disruptions by switching fuel sources.Petroleum Refineries (Residual Fuels & Petcoke):Petroleum Connection (Incoming): Refines crude oil streams into lighter transportation fuels while outputting heavy residual bottom fractions.Electric Power Connection (Outgoing): Supplies petroleum coke (petcoke) and heavy fuel oil directly to thermal power plants and industrial co-generation units.Electrified Oilfield & Midstream Infrastructure:Electric Power Connection (Incoming): Consumes bulk electricity drawn directly from regional transmission grids (e.g., ERCOT, SPP).Petroleum Connection (Outgoing): Powers electric drilling rigs, hydraulic fracturing fleets, and pipeline pump/compressor stations required to keep crude oil moving through midstream networks."

    ],

}

In [18]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [19]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.0          0.991839                0.0             0.0
Vector RAG           1.0          0.000000                0.0             0.0


Question 8

In [24]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How would severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [[
            "barrels with more than 14 mb/d of oil now shut in, an unprecedented supply shock. The current supply-demand gap is significantly smaller, however, as the market was already in surplus heading into the crisis while producers and consumers alike are responding to market signals. On the supply side, Saudi Arabia and the UAE have successfully redirected some exports to terminals loading outside of the Strait. At the same time, stocks from commercial and government strategic storage sites in consuming countries are flowing into markets  to offset part of the losses. Observed global inventories, including oil on water, were drawn down by 250 mb over March and April , or 4 mb/d. Producers outside of the Middle East also pushed output higher and lifted exports to record levels in response to the crisis. Indeed, 2026 supply growth expectations from the Americas have been revised up by"
    ]],
    "reference": [
        "Acute warehousing limitations within Middle Eastern harbors, including Fujairah in the UAE, would modify the reactive sequence by interrupting the movement and provision of hydrocarbon goods and unrefined petroleum. In particular, hydrocarbon outputs originating from Fujairah influence the total availability, are manufactured within processing plants, and get conveyed through tubes, flatboats, and freighters toward unrefined petroleum within the Strategic Petroleum Reserve. Furthermore, unrefined petroleum alongside alternative commodities originating from the Middle East/UAE get shipped through flatboats toward passage bottlenecks such as the Panama Canal. Consequently, warehousing limitations within those harbors would disrupt those processing, conveyance (through tubes, flatboats, and freighters), and provision routes, eventually affecting both the Strategic Petroleum Reserve alongside passage across crucial bottlenecks."
    ],
}

graph_rag_data = {
    "question": [
        "How would severe storage constraints in Middle Eastern ports alter the causal chain during a transit chokepoint closure?"
    ],
    "answer": [
       "Severe storage constraints in Middle Eastern ports, such as Fujairah in the UAE, would alter the causal chain by disrupting the flow and supply of petroleum products and crude oil. Specifically, petroleum products from Fujairah affect the overall supply, are produced in refineries, and are transported via pipelines, barges, and tankers to crude oil in the Strategic Petroleum Reserve. Additionally, crude oil and other products from the Middle East/UAE are transported via barges to transit chokepoints like the Panama Canal. Therefore, storage constraints in these ports would interrupt these refining, transport (via pipelines, barges, and tankers), and supply pathways, ultimately impacting both the Strategic Petroleum Reserve and transit through key chokepoints."
    ],
    "contexts": 
        [[
       "Fujairah is located in Middle East. Middle East is located in Crude Oil. Crude Oil is transported via Barge. Barge is located in Panama Canal. Crude Oil is transported via Tanker. Tanker is located in Panama Canal. Fujairah is located in Petroleum Product. Petroleum Product affects the supply of Supply. Supply affects the supply of Crude Oil. Crude Oil is located in Strategic Petroleum Reserve. Crude Oil is transported via Railroad. Petroleum Product supplies to Crude Oil. Petroleum Product is transported via Pipelines. Pipelines are transported via Crude Oil. Petroleum Product is transported via Barge. Barge is transported via Crude Oil."
        ]],
    "reference": [
        "Acute warehousing limitations within Middle Eastern harbors, including Fujairah in the UAE, would modify the reactive sequence by interrupting the movement and provision of hydrocarbon goods and unrefined petroleum. In particular, hydrocarbon outputs originating from Fujairah influence the total availability, are manufactured within processing plants, and get conveyed through tubes, flatboats, and freighters toward unrefined petroleum within the Strategic Petroleum Reserve. Furthermore, unrefined petroleum alongside alternative commodities originating from the Middle East/UAE get shipped through flatboats toward passage bottlenecks such as the Panama Canal. Consequently, warehousing limitations within those harbors would disrupt those processing, conveyance (through tubes, flatboats, and freighters), and provision routes, eventually affecting both the Strategic Petroleum Reserve alongside passage across crucial bottlenecks."
    ],
}

In [25]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [ ]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.0          0.924941                1.0             0.0
Vector RAG           0.2          0.000000                0.0             0.0


Question 9

In [27]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "If Permian Basin oil drilling declines sharply, how does the graph update its prediction for Henry Hub gas prices?"
    ],
    "answer": [
       "Based on the provided context, the relationship between oil drilling, associated natural gas production, and Henry Hub prices works as follows: first, for the associated natural gas link, increased oil production concurrently produces more associated natural gas, and second, regarding the impact on prices, when oil production is high, the extra associated natural gas increases overall natural gas production and inventories, which lowers the forecast for Henry Hub spot prices. Regarding how the prediction would update, if Permian Basin oil drilling declines sharply, there would be a corresponding decrease in associated natural gas production. With less natural gas being produced and held in inventory, there would be less downward pressure (and conversely, more upward pressure) on prices, and as a result, the prediction for Henry Hub gas prices would be revised upward (meaning higher expected prices than the current forecast of $3.46/MMBtu in 2027)."
         ],
    "contexts": [[
            "Gulf Coast LNG export facilities. Together, these regions produce enough natural gas to keep inventories above the five-year average and limit upward pressure on Henry Hub prices."
    ]],
    "reference": [
        "The forecast is revised depending on the sequence wherein the Permian Basin yields Natural Gas, which immediately influences the Henry Hub spot valuation situated at Henry Hub."
    ],
}

graph_rag_data = {
    "question": [
        "If Permian Basin oil drilling declines sharply, how does the graph update its prediction for Henry Hub gas prices?"
    ],
    "answer": [
       "The prediction is updated based on the chain where the Permian Basin produces Natural Gas, which directly impacts the Henry Hub Spot Price located in Henry Hub."
    ],
    "contexts": 
        [[
       "The Permian Basin produces Natural Gas. Natural Gas impacts the price of Henry Hub Spot Price. Henry Hub Spot Price is located in Henry Hub.",
    ]],
    "reference": [
        "The forecast is revised depending on the sequence wherein the Permian Basin yields Natural Gas, which immediately influences the Henry Hub spot valuation situated at Henry Hub."
    ],
}

In [28]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [29]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.5          0.796142                1.0             0.0
Vector RAG           0.0          0.926442                0.0             0.0


Question 10 --- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How would a sudden collapse in global crude oil demand alter the relationships between OPEC production and OECD tax revenues?"
    ],
    "answer": [
       "Based on the provided context, there is no direct discussion or analysis of how a sudden collapse in global crude oil demand would alter the relationships between OPEC production and OECD tax revenues. However, we can infer the structural vulnerability of both components based on the data provided for the current relationship. Regarding **revenue disparity**, the estimated annual average OECD retail revenues (for 2020–2024) are **$2.571 trillion/year**, whereas annual average OPEC oil revenues are **$0.599 trillion/year**—resulting in a difference of **$1.972 trillion/year**. Regarding the **composite barrel share (OECD in 2023)**, **government tax** is $113.48, **crude price (to oil producers)** is $80.21, and **industry margin** is $52.73. As an analytical conclusion, in the event of a sudden demand collapse, **OPEC revenues** would likely fall precipitously because they are directly tied to the crude oil price ($80.21 of the composite barrel) and export volumes, while **OECD tax revenues** would also decline due to lower consumption volumes. However, because government taxes represent the largest single component of the composite barrel ($113.48)—and many retail fuel taxes in OECD countries are volume-based (fixed per gallon/liter) rather than value-based—the OECD tax mechanism provides a cushion, meaning OECD government revenues would likely prove more resilient per barrel than OPEC's crude revenues, further widening the existing $1.972 trillion revenue gap."
    ],
    "contexts": [[
            "average prices.Annual average revenues Tax Crude price Industry m argin E s t im a t e d  a n n u a l average OECD retail revenues E s t im a t e d  a n n u a laverage OPEC oil revenues %$bn/year OECD average OECD average Comparison of composite barrel (OECD average)/.null T a x C ru d e  p ric e"
    ]],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "How would a sudden collapse in global crude oil demand alter the relationships between OPEC production and OECD tax revenues?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'p1': [{'id': 'Opec'}, 'AFFECTS_SUPPLY_OF', {'id': 'Crude Oil'}, 'PRODUCED_IN', {'id': 'Padd'}, 'LOCATED_IN', {'id': 'Product Supplied'}, 'MEASURED_IN', {'id': 'Petroleum Demand'}], 'p2': [{'id': 'Petroleum Demand'}, 'MEASURED_IN', {'id': 'Product Supplied'}, 'LOCATED_IN', {'id': 'Padd'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'Oecd'}, 'LOCATED_IN', {'id': 'Demand'}], 'p2': [{'id': 'Demand'}, 'LOCATED_IN', {'id': 'Oecd'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'Crude'}, 'AFFECTS_SUPPLY_OF', {'id': 'Middle East Conflict'}, 'AFFECTS', {'id': 'Supply/Demand Forecasts'}], 'p2': [{'id': 'Supply/Demand Forecasts'}, 'AFFECTS', {'id': 'Middle East Conflict'}, 'AFFECTS', {'id': 'Petroleum Product'}, 'PRODUCES', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'Global Oil Demand'}], 'p2': [{'id': 'Global Oil Demand'}, 'FORECASTS_OUTLOOK', {'id': 'U.S. Energy Information Administration'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'AFFECTS_SUPPLY_OF', {'id': 'Oil'}, 'AFFECTS', {'id': 'Demand Saving Measures'}], 'p2': [{'id': 'Demand Saving Measures'}, 'AFFECTS', {'id': 'Oil'}, 'PRODUCES', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'China'}, 'LOCATED_IN', {'id': 'Pump Prices'}, 'AFFECTS', {'id': 'Gasoline Demand'}], 'p2': [{'id': 'Gasoline Demand'}, 'AFFECTS', {'id': 'Pump Prices'}, 'AFFECTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'LOCATED_IN', {'id': 'World'}, 'LOCATED_IN', {'id': 'China'}, 'LOCATED_IN', {'id': 'China Total Oil Demand'}], 'p2': [{'id': 'China Total Oil Demand'}, 'LOCATED_IN', {'id': 'China'}, 'LOCATED_IN', {'id': 'World'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'LOCATED_IN', {'id': 'World'}, 'LOCATED_IN', {'id': 'India'}, 'LOCATED_IN', {'id': 'India Total Oil Demand'}], 'p2': [{'id': 'India Total Oil Demand'}, 'LOCATED_IN', {'id': 'India'}, 'LOCATED_IN', {'id': 'World'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'India'}, 'LOCATED_IN', {'id': 'Oil Demand Growth'}], 'p2': [{'id': 'Oil Demand Growth'}, 'LOCATED_IN', {'id': 'India'}, 'LOCATED_IN', {'id': 'Pump Prices'}, 'AFFECTS', {'id': 'Fuel Taxes'}]}, {'p1': [{'id': 'Opec'}, 'FORECASTS_OUTLOOK', {'id': 'Oil Demand'}], 'p2': [{'id': 'Oil Demand'}, 'LOCATED_IN', {'id': 'Oecd Americas'}, 'LOCATED_IN', {'id': 'United States'}, 'ENACTS', {'id': 'Fuel Taxes'}]}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question 11 --- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How do sub-graph connections shift if electric power utilities substitute coal for natural gas during summer peak loads?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "How do sub-graph connections shift if electric power utilities substitute coal for natural gas during summer peak loads?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'p': [{'id': 'Coal'}, 'SUPPLIES_TO', {'id': 'Electric Power'}, 'SUPPLIES_TO', {'id': 'Natural Gas'}]}, {'p': [{'id': 'Coal'}, 'SUPPLIES_TO', {'id': 'Electric Power Sector'}, 'SUPPLIES_TO', {'id': 'Natural Gas'}]}, {'p': [{'id': 'Coal'}, 'AFFECTS', {'id': 'Power Generation Fuel Costs'}, 'AFFECTS', {'id': 'Natural Gas'}]}, {'p': [{'id': 'Coal'}, 'SUBJECT_TO', {'id': 'Power Generation Fuel Costs'}, 'AFFECTS', {'id': 'Natural Gas'}]}, {'p': [{'id': 'Coal'}, 'AFFECTS', {'id': 'Power Generation Fuel Costs'}, 'SUBJECT_TO', {'id': 'Natural Gas'}]}, {'p': [{'id': 'Coal'}, 'SUBJECT_TO', {'id': 'Power Generation Fuel Costs'}, 'SUBJECT_TO', {'id': 'Natural Gas'}]}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question 12

In [30]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "List all incoming and outgoing connections for the Brent pricing benchmark node."
    ],
    "answer": [
       "Based on the provided context, there is no mention of the Brent pricing benchmark node or any incoming and outgoing connections associated with it. Therefore, I do not have the information to answer this question."
    ],
    "contexts": [[
            " Americas Europe Asia Oceania Total OECD  NON-OECD SUPPLY Eurasia Europe China Other Asia Latin America Middle East "
    ]],
    "reference": [
        "Brent is situated within the North Sea. Brent is positioned in Rotterdam. Brent is quantified in US$/B. ICE manages Brent. OPEC predicts the prospects for Brent. Brent is extracted within the North Sea. Brent is generated across Northern Europe. Brent influences the valuation of North Sea Dated. Administered Capital Net Long Holdings affect the cost of Brent."
    ],
}

graph_rag_data = {
    "question": [
        "List all incoming and outgoing connections for the Brent pricing benchmark node."
    ],
    "answer": [
       "The incoming and outgoing connections for the Brent pricing benchmark node are outgoing connections (from Brent), including LOCATED_IN North Sea, Rotterdam, MEASURED_IN Us$/B, PRODUCED_IN North Sea, Northern Europe, and IMPACTS_PRICE North Sea Dated, as well as incoming connections (to Brent), including OPERATES Ice, FORECASTS_OUTLOOK Opec, and IMPACTS_PRICE Managed Money Net Long Positions."
    ],
    "contexts": 
        [[
        "Brent is located in North Sea. Brent is located in Rotterdam. Brent is measured in Us$/B. Ice operates Brent. Opec forecasts the outlook for Brent. Brent is produced in North Sea. Brent is produced in Northern Europe. Brent impacts the price of North Sea Dated. Managed Money Net Long Positions impacts the price of Brent."
    ]],
    "reference": [
        "Brent is situated within the North Sea. Brent is positioned in Rotterdam. Brent is quantified in US$/B. ICE manages Brent. OPEC predicts the prospects for Brent. Brent is extracted within the North Sea. Brent is generated across Northern Europe. Brent influences the valuation of North Sea Dated. Administered Capital Net Long Holdings affect the cost of Brent."
    ],
}

In [31]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [33]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           1.0          0.966747                1.0             1.0
Vector RAG           1.0          0.000000                0.0             0.0


Question 13

In [37]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "Which specific entities and relationships form the local subgraph for the Permian Region?"
    ],
    "answer": [
       "For entities, these include 1. Permian region (The primary entity of interest, appearing in multiple data series), 2. Permian formations (A closely related formation-level entity), 3. Peer Regions (Entities grouped in the same regional datasets): Bakken region, Eagle Ford region, Haynesville region, Appalachia region, and Rest of Lower 48 States, 4. Peer Formations (Entities grouped in the same formation-level dataset): Eagle Ford formation, Mississippian formation, and Niobrara Codell formation, and 5. Numerical Metrics/Data Series (The specific data arrays mapped to each entity). Under relationships, for 1. Regional Grouping (Co-occurrence), the Permian region shares a sibling relationship with the Bakken region, Eagle Ford region, and Haynesville region across multiple data series, including Series 1 (Values around 440–458), Series 3 (Negative values around -428 to -453), and Series 4 (Values around 6.41–7.34, which also includes the Appalachia region and the Rest of Lower 48 States), followed by 2. Formation Grouping."
    ],
    "contexts": [[
            "Appalachia region Bakken region Eagle Ford region Haynesville region Permian region Rest of Lower 48 States "
    ]],
    "reference": [
        "The Permian Region is a major energy-producing area located in the Lower 48 States of the United States that directly affects the nation's Total Primary Supply. Resource Production: The region produces both Natural Gas and Crude Oil, including specific tracking of crude oil production from newly completed wells. Drilling Metrics: Key operational measurements tracked in the region include New Wells Drilled, New Wells Drilled Per Rig, and Cumulative Drilled But Uncompleted Wells.Forecasting: The U.S. Energy Information Administration (EIA) monitors these metrics and forecasts the ongoing outlook for the region."
    ],
}

graph_rag_data = {
    "question": [
        "Which specific entities and relationships form the local subgraph for the Permian Region?"
    ],
    "answer": [
       "The Permian Region is located in the United States. The Permian Region is located in the Lower 48 States. Cumulative Drilled But Uncompleted Wells are located in the Permian Region. New Wells Drilled are measured in the Permian Region. New Wells Drilled Per Rig are measured in the Permian Region. The Permian Region affects the supply of Total Primary Supply. The U.S. Energy Information Administration forecasts the outlook for the Permian Region. Crude Oil is produced in the Permian Region. Crude Oil Production From Newly Completed Wells is produced in the Permian Region. The Permian Region produces Natural Gas."
    ],
    "contexts": 
        [[
            "The Permian Region is located in the United States. The Permian Region is located in the Lower 48 States. Cumulative Drilled But Uncompleted Wells are located in the Permian Region. New Wells Drilled are measured in the Permian Region. New Wells Drilled Per Rig are measured in the Permian Region. The Permian Region affects the supply of Total Primary Supply. The U.S. Energy Information Administration forecasts the outlook for the Permian Region. Crude Oil is produced in the Permian Region. Crude Oil Production From Newly Completed Wells is produced in the Permian Region. The Permian Region produces Natural Gas.",
    ]],    
    "reference": [
        "The Permian Region is a major energy-producing area located in the Lower 48 States of the United States that directly affects the nation's Total Primary Supply. Resource Production: The region produces both Natural Gas and Crude Oil, including specific tracking of crude oil production from newly completed wells. Drilling Metrics: Key operational measurements tracked in the region include New Wells Drilled, New Wells Drilled Per Rig, and Cumulative Drilled But Uncompleted Wells.Forecasting: The U.S. Energy Information Administration (EIA) monitors these metrics and forecasts the ongoing outlook for the region."
    ]
}

In [38]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [39]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           1.0          0.812634                1.0             1.0
Vector RAG           0.0          0.909330                0.0             0.0


Question 14 ---- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "Which entities contribute to retail pump prices, and how do their structural weightings differ between consuming and producing regions?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "Which entities contribute to retail pump prices, and how do their structural weightings differ between consuming and producing regions?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'Contributor': 'Export Restrictions', 'ContributorType': ['Policy'], 'StructuralWeight': None, 'Region': None}, {'Contributor': 'Electricity', 'ContributorType': ['Commodity'], 'StructuralWeight': 'Prices To Ultimate Customers', 'Region': 'East South Central'}, {'Contributor': 'Electricity', 'ContributorType': ['Commodity'], 'StructuralWeight': 'Prices To Ultimate Customers', 'Region': 'West South Central'}, {'Contributor': 'Electricity', 'ContributorType': ['Commodity'], 'StructuralWeight': 'Prices To Ultimate Customers', 'Region': 'Mountain'}, {'Contributor': 'Electricity', 'ContributorType': ['Commodity'], 'StructuralWeight': 'Prices To Ultimate Customers', 'Region': 'Pacific Contiguous'}, {'Contributor': 'Electricity', 'ContributorType': ['Commodity'], 'StructuralWeight': 'Prices To Ultimate Customers', 'Region': 'Alaska And Hawaii'}, {'Contributor': 'Natural Gas', 'ContributorType': ['Commodity'], 'StructuralWeight': None, 'Region': 'United States'}, {'Contributor': 'Natural Gas', 'ContributorType': ['Commodity'], 'StructuralWeight': None, 'Region': 'U.S.'}, {'Contributor': 'Natural Gas', 'ContributorType': ['Commodity'], 'StructuralWeight': None, 'Region': 'Alaska'}, {'Contributor': 'Natural Gas', 'ContributorType': ['Commodity'], 'StructuralWeight': None, 'Region': 'Gulf Of America'}]",
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question 15

In [40]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "Compare the regulatory and fiscal influence of consuming governments against the market influence of major oil-producing cartels."
    ],
    "answer": [
       "While public perception often attributes high fuel costs and massive wealth generation to producing cartels like OPEC, the data shows this is a misconception. Through aggressive fiscal policies and retail taxation, consuming governments (OECD) exert the dominant economic influence, extracting far more revenue from a barrel of oil than the countries that produce it."
    ],
    "contexts": [[
            "Comparison of composite barrel (OECD average) An n u a l average OECD retail revenues An n u a l average OPEC oil revenues $tn/year % change in a composite  barrel shares  Estimated annual average revenues (OECD Estimated annual average revenues (OECD Composite barrel components (OECD in Difference $1.972 trillion 2.571 0.599 (OECD retail vs. OPEC oil, 2020–24)"
    ]],
    "reference": [
        "The Russian Government enacts Export Restrictions, which impacts the price of Commodity Price. The Government enacts Government Subsidies, which impacts the price of Consumer Fuel Prices. Central Bank Of Brazil enacts Monetary Policy, which affects Inflation. Central Bank Of Brazil enacts Monetary Policy, which affects Growth Dynamics. Central Bank Of Brazil enacts Monetary Policy, which affects Interest Rates. Fomc enacts Monetary Policy, which affects Inflation. Fomc enacts Monetary Policy, which affects Growth Dynamics. Fomc enacts Monetary Policy, which affects Interest Rates. Ecb enacts Monetary Policy, which affects Inflation. Ecb enacts Monetary Policy, which affects Growth Dynamics. Across all contexts, Iea produces Iea Subscription Data Services."
    ],
}

graph_rag_data = {
    "question": [
         "Compare the regulatory and fiscal influence of consuming governments against the market influence of major oil-producing cartels."
    ],
    "answer": [
       "While regulatory bodies and governments shape the market through compliance, reporting, and collective actions, cartels like OPEC wield direct control over supply levels."
    ],
    "contexts": 
        [[
           "The Russian Government enacts Export Restrictions, which impacts the price of Commodity Price. The Government enacts Government Subsidies, which impacts the price of Consumer Fuel Prices. Central Bank Of Brazil enacts Monetary Policy, which affects Inflation. Central Bank Of Brazil enacts Monetary Policy, which affects Growth Dynamics. Central Bank Of Brazil enacts Monetary Policy, which affects Interest Rates. Fomc enacts Monetary Policy, which affects Inflation. Fomc enacts Monetary Policy, which affects Growth Dynamics. Fomc enacts Monetary Policy, which affects Interest Rates. Ecb enacts Monetary Policy, which affects Inflation. Ecb enacts Monetary Policy, which affects Growth Dynamics. Across all contexts, Iea produces Iea Subscription Data Services."
    ]],
    "reference": [
        "The Russian Government enacts Export Restrictions, which impacts the price of Commodity Price. The Government enacts Government Subsidies, which impacts the price of Consumer Fuel Prices. Central Bank Of Brazil enacts Monetary Policy, which affects Inflation. Central Bank Of Brazil enacts Monetary Policy, which affects Growth Dynamics. Central Bank Of Brazil enacts Monetary Policy, which affects Interest Rates. Fomc enacts Monetary Policy, which affects Inflation. Fomc enacts Monetary Policy, which affects Growth Dynamics. Fomc enacts Monetary Policy, which affects Interest Rates. Ecb enacts Monetary Policy, which affects Inflation. Ecb enacts Monetary Policy, which affects Growth Dynamics. Across all contexts, Iea produces Iea Subscription Data Services."
    ]
}

In [41]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [43]:
# Execute evaluations
vector_results_df = evaluate_pipeline(vector_rag_data, "Vector RAG")
graph_results_df = evaluate_pipeline(graph_rag_data, "Graph RAG")

# Combine results for comparative analysis
combined_df = pd.concat([vector_results_df, graph_results_df], ignore_index=True)

# Generate aggregate summary table
metrics_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall",
]
summary_df = combined_df.groupby("pipeline")[metrics_cols].mean().reset_index()

print("--- Aggregate RAG Benchmark Scores ---")
print(summary_df.to_string(index=False))

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


--- Aggregate RAG Benchmark Scores ---
  pipeline  faithfulness  answer_relevancy  context_precision  context_recall
 Graph RAG           0.0          0.908425                0.0             1.0
Vector RAG           0.0          0.916473                0.0             0.0


Question 16 --- ?

In [ ]:
# Initialize judge LLM and embedding model
eval_llm = llm
eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "When queried about the local economic impact of Permian Region drilling, how does GraphRAGs local entity neighborhood search compare to a global community summary query on U.S. energy market trends?"
    ],
    "answer": [
       "Based on the provided context, there is no explicit graph, network structure, or defined nodes and relationships. However, looking at the data tables (which resemble metrics from the EIA Drilling Productivity Report), the **Permian region** is associated with two distinct data series: 1. **Positive Metrics/Inputs (potentially rigs or new-well production):** Values: 443, 458, 454, 449, 458, and a later value of 451 (with remaining periods represented by -). 2. **Negative Metrics/Outputs (typically legacy production decline):*** Values: -437.2, -428.1, -430.2, -432.1, -453.8, and a later value of -431.9 (with remaining periods represented by -).If you are referring to a specific conceptual model or a diagram not fully rendered in the text, those relationships are not defined in the provided context."
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "When queried about the local economic impact of Permian Region drilling, how does GraphRAGs local entity neighborhood search compare to a global community summary query on U.S. energy market trends?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Permian Region'}, 'LOCATED_IN', {'id': 'United States'}), 'n': {'id': 'United States'}, 'u': {'id': 'United States Energy Market'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Permian Region'}, 'LOCATED_IN', {'id': 'United States'}), 'n': {'id': 'United States'}, 'u': {'id': 'United States'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Permian Region'}, 'LOCATED_IN', {'id': 'Lower 48 States'}), 'n': {'id': 'Lower 48 States'}, 'u': {'id': 'United States Energy Market'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Permian Region'}, 'LOCATED_IN', {'id': 'Lower 48 States'}), 'n': {'id': 'Lower 48 States'}, 'u': {'id': 'United States'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Cumulative Drilled But Uncompleted Wells'}, 'LOCATED_IN', {'id': 'Permian Region'}), 'n': {'id': 'Cumulative Drilled But Uncompleted Wells'}, 'u': {'id': 'United States Energy Market'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'Cumulative Drilled But Uncompleted Wells'}, 'LOCATED_IN', {'id': 'Permian Region'}), 'n': {'id': 'Cumulative Drilled But Uncompleted Wells'}, 'u': {'id': 'United States'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'New Wells Drilled'}, 'MEASURED_IN', {'id': 'Permian Region'}), 'n': {'id': 'New Wells Drilled'}, 'u': {'id': 'United States Energy Market'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'New Wells Drilled'}, 'MEASURED_IN', {'id': 'Permian Region'}), 'n': {'id': 'New Wells Drilled'}, 'u': {'id': 'United States'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'New Wells Drilled Per Rig'}, 'MEASURED_IN', {'id': 'Permian Region'}), 'n': {'id': 'New Wells Drilled Per Rig'}, 'u': {'id': 'United States Energy Market'}}, {'r': {'id': 'Permian Region'}, 'rel': ({'id': 'New Wells Drilled Per Rig'}, 'MEASURED_IN', {'id': 'Permian Region'}), 'n': {'id': 'New Wells Drilled Per Rig'}, 'u': {'id': 'United States'}}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

Question 17 -- ?

In [ ]:
# Initialize judge LLM and embedding model
#eval_llm = llm
#eval_embeddings = embedder

# Benchmark datasets for Vector RAG and Graph RAG
vector_rag_data = {
    "question": [
        "How does community-based GraphRAG aggregate information about U.S. refined petroleum exports across distinct regional sub-communities?"
    ],
    "answer": [
       "Based on the provided context, which consists of tabular data, there are no explicitly defined graph-based "incoming" or "outgoing" structural relationships. However, we can identify the relationships based on the groupings and numerical data (positive and negative flows) associated with the **Permian region** node: ### 1. Peer Group Relationships (Sibling Nodes) The Permian region belongs to the same category/grouping as the following regional nodes: *   **Bakken region** *   **Eagle Ford region** *   **Haynesville region** --### 2. Numerical Data Flows (Incoming and Outgoing Values)If we interpret the positive values as incoming metrics and negative values as outgoing metrics associated with the Permian region: *   **'Incoming' (Positive) Data Series:** *   Values: `443`, `458`, `454`, `449`, `458` (with an average/summary value of `451`) *   **'Outgoing' (Negative) Data Series:** *   Values: `-437.2`, `-428.1`, `-430.2`, `-432.1`, `-453.8` (with an average/summary value of `-431.9`)"
    ],
    "contexts": [
            "Americas  0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.2 0.1 0.1 0.1\nEurope 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.1 0.0 0.0 0.0\nAsia Oceania 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nTotal OECD         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.1 0.1 0.1 0.1 0.1\nNON-OECD SUPPLY\nEurasia 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nEurope                         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nChina                          0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nOther Asia                     0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0\nLatin America             0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.3 0.2 0.0 0.1\nMiddle East         0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 0.0 -0.3 -0.7 -0.2 -0.3"
    ],
    "ground_truth": [
        "Drug A side effects include nausea and fatigue. When combined with Drug B, it increases liver toxicity risks."
    ],
}

graph_rag_data = {
    "question": [
        "How does community-based GraphRAG aggregate information about U.S. refined petroleum exports across distinct regional sub-communities?"
    ],
    "answer": [
       "The direct incoming and outgoing relationships centered on the Permian Region node are:\n\n**Outgoing Relationships:**\n*   Permian Region is **LOCATED_IN** the United States.\n*   Permian Region is **LOCATED_IN** the Lower 48 States.\n*   Permian Region **AFFECTS_SUPPLY_OF** Total Primary Supply.\n*   Permian Region **PRODUCES** Natural Gas.\n\n**Incoming Relationships:**\n*   Cumulative Drilled But Uncompleted Wells is **LOCATED_IN** the Permian Region.\n*   New Wells Drilled is **MEASURED_IN** the Permian Region.\n*   New Wells Drilled Per Rig is **MEASURED_IN** the Permian Region.\n*   U.S. Energy Information Administration **FORECASTS_OUTLOOK** the Permian Region.\n*   Crude Oil is **PRODUCED_IN** the Permian Region.\n*   Crude Oil Production From Newly Completed Wells is **PRODUCED_IN** the Permian Region."
    ],
    "contexts": 
        [
            "[{'Region': 'Nationwide', 'RefinedPetroleum': 'Petroleum Products', 'AggregateMetric': 'Ending Stocks'}, {'Region': 'Oecd', 'RefinedPetroleum': 'Petroleum Products', 'AggregateMetric': 'Ending Stocks'}, {'Region': 'Europe', 'RefinedPetroleum': 'Petroleum Products', 'AggregateMetric': 'Ending Stocks'}, {'Region': 'Oecd Europe', 'RefinedPetroleum': 'Petroleum Products', 'AggregateMetric': 'Ending Stocks'}, {'Region': 'United States', 'RefinedPetroleum': 'Petroleum Products', 'AggregateMetric': 'Ending Stocks'}]"
    ],
}

In [ ]:
def evaluate_pipeline(data: dict, pipeline_name: str) -> pd.DataFrame:
    """Converts input data to a HuggingFace Dataset and runs Ragas evaluation."""
    dataset = Dataset.from_dict(data)

    metrics = [
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ]

    results = evaluate(
        dataset=dataset,
        metrics=metrics,
        llm=eval_llm,
        embeddings=eval_embeddings,
    )

    df = results.to_pandas()
    df["pipeline"] = pipeline_name
    return df

In [20]:
import os
from ragas import EvaluationDataset, SingleTurnSample, evaluate
from ragas.metrics import Faithfulness, answer_relevancy,context_precision,context_recall

# 1. Ensure OpenAI API key is set
#os.environ["OPENAI_API_KEY"] = "your-openai-api-key"

# 2. Simulate outputs from Microsoft GraphRAG (Local or Global Search)
#query = "What is the Brent benchmark?"
#response = "The Brent benchmark is a major trading classification of sweet light crude oil."

# GraphRAG returns context as a single formatted string block
#graphrag_context_str = (
    #"The Brent benchmark is a major trading classification of sweet light crude oil. "
    #"It serves as a benchmark price for purchases of oil worldwide. There are no listed chokepoints."
#)

# 3. Format context into a list of strings (list[str])
# Option A: Wrap the entire block in a single-element list
#retrieved_contexts = [graphrag_context_str]

# Option B: Split text into separate context chunks if multiline
# retrieved_contexts = [chunk.strip() for chunk in graphrag_context_str.split("\n\n") if chunk.strip()]

query = "How are the pricing benchmarks (Brent and Henry Hub) grouped into graph communities relative to global supply chokepoints versus domestic production basins?"
response = "Global Supply Chokepoints & Brent (Global Oil Benchmarks): Brent crude (along with other crude benchmarks like North Sea Dated, WTI, and Dubai) is grouped with global supply disruptions and chokepoints. Specifically, the text highlights that the de facto closure of the Strait of Hormuz (a major world oil transit chokepoint) and military action drive heightened volatility and directly impact the Brent crude oil spot price. Domestic Production Basins & Henry Hub (U.S. Natural Gas Benchmark): Henry Hub is grouped with domestic production basins and regional infrastructure. The text notes that domestic production regions near the Gulf Coast LNG export facilities produce enough natural gas to keep inventories above the five-year average, which limits upward pressure on Henry Hub natural gas prices."
retrieved_contexts =  "U.S. net trade of hydrocarbon gas liquids (HGL) million barrels per day net trade propane ethane natural gasoline butanes forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026 net imports net exports Henry Hub natural gas price and NYMEX futures price dollars per million British thermal units Note: Futures curve is the average settlement price for five trading days ending May 7, 2026.  Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026, Bloomberg L.P., and LSEG Data STEO forecast NYMEX futures price Henry Hub spot price U.S. natural gas prices dollars per thousand cubic feet forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026" 
reference =  "U.S. net trade of hydrocarbon gas liquids (HGL) million barrels per day net trade propane ethane natural gasoline butanes forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026 net imports net exports Henry Hub natural gas price and NYMEX futures price dollars per million British thermal units Note: Futures curve is the average settlement price for five trading days ending May 7, 2026.  Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026, Bloomberg L.P., and LSEG Data STEO forecast NYMEX futures price Henry Hub spot price U.S. natural gas prices dollars per thousand cubic feet forecast Data source: U.S. Energy Information Administration, Short-Term Energy Outlook, June 2026" 
retrieved_contexts = [retrieved_contexts]

# 4. Create SingleTurnSample object
sample = SingleTurnSample(
    user_input=query,
    response=response,
    retrieved_contexts=retrieved_contexts,  # Must be list[str], not str
    reference = reference
)

# 5. Build RAGAS Evaluation Dataset
dataset = EvaluationDataset(samples=[sample])

# 6. Evaluate metrics
metrics = [faithfulness,
        answer_relevancy,
        context_precision,
        context_recall]
results = evaluate(dataset=dataset, metrics=metrics,llm=eval_llm,
        embeddings=eval_embeddings)

# 7. Output results
print(results)

/var/folders/dr/mrkz6zcx1tz_z87q4dtf9qym0000gn/T/ipykernel_94840/4217735161.py:3: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, answer_relevancy,context_precision,context_recall
/var/folders/dr/mrkz6zcx1tz_z87q4dtf9qym0000gn/T/ipykernel_94840/4217735161.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import Faithfulness, answer_relevancy,context_precision,context_recall
/var/folders/dr/mrkz6zcx1tz_z87q4dtf9qym0000gn/T/ipykernel_94840/4217735161.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.me

Evaluating:   0%|          | 0/4 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


{'faithfulness': 0.0000, 'answer_relevancy': 0.9202, 'context_precision': 0.0000, 'context_recall': 1.0000}
